# Part e: Classification on MNIST


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # add Code/ so `src` can be imported

In [ ]:
import numpy as np
from src.data import load_mnist, one_hot, stratified_kfold_indices
from src.metrics import accuracy, confusion_matrix_multiclass, per_class_from_confusion
from src.network import NeuralNetwork
from src.functions import softmax, ReLU, ReLU_der, identity_der, cross_entropy, softmax_cross_entropy_der
from src.optimizers import Adam

In [ ]:
x_train, y_train, x_test, y_test = load_mnist("../data/mnist.npz") # loading the mnist dataset, flattening the image pixel to shape (784,)
print(x_train.shape, y_train.shape, x_test.shape, y_test.shape) # checking the shapes of training set, test set and true digits

rng = np.random.default_rng(2026) # setting a seed for reproducibility

# for every digit, find 1000 random positions of images that is this digit. Join the 1000 image positions for every digit in one long array
# This makes every digit equally represented
sub = np.concatenate([rng.choice(np.where(y_train == c)[0], 1000, replace=False) for c in range(10)])
X_sub, y_sub = x_train[sub], y_train[sub] # pick out these randomly chosen images by using their positions (sub)

# should give shape (10000, 784) since 1000 * 10 digit images and each flattened image has 784 pixels. 
# Bincount counts how many times each digit appear, should be 1000 of each digit:
print(X_sub.shape, np.bincount(y_sub))  # check shapes  

In [ ]:
logreg = NeuralNetwork(784, [10], [softmax], [identity_der],
                       cross_entropy, softmax_cross_entropy_der, Adam(gamma=0.001))
logreg.train(X_sub, one_hot(y_sub, 10), epochs=5, batch_size=100, rng=np.random.default_rng(2026))
print("training accuracy:", accuracy(logreg.predict(X_sub), one_hot(y_sub, 10)))

# Logistic regression network, 784 -> 10 (no hidden layers), softmax output + cross entropy
'''
Functions used:

softmax turns the 10 outputs into probabilities that sum to 1.
cross_entropy is the cost function for classification.
softmax_cross_entropy_der is the derivative used in backpropagation for the last layer. 
identity_der is a derivative that is always 1
The Adam optimizer updates the weights during training.
'''
# Creating the network
logreg = NeuralNetwork(784, [10], [softmax], [identity_der],      # identity_der: the softmax derivative is already in softmax_cross_entropy_der
                       cross_entropy, softmax_cross_entropy_der, Adam(gamma=0.001))

# Training the network with stochastic gradient descent, gives the training cost after each epoch
history = logreg.train(x_tr, Y_tr, epochs=5, batch_size=100, rng=np.random.default_rng(2026))


# Finding the predicted numbers for each image in the validation dataset
# logreg.predict(x_val) gives 10 probabilities for each of the 10 000 validation images, with shape (10000, 10).
# np.argmax(..., axis=1) picks, for each image, the position of the largest probability, i.e. the predicted digit. The result has shape (10000,).
pred_val = np.argmax(logreg.predict(x_val), axis=1)    # predicted digit = the output with the highest probability

# compares the predicted digits with the true digits y_val, and prints the fraction that are correct
print("validation accuracy for logistic regression:", accuracy(y_val, pred_val))

# Network with one hidden layer of 100 ReLU nodes: 784 -> 100 -> 10
# same principle as for logistic regression, but adding activation function ReLU for the hidden layer and outputs 100 and 10

net_layer = NeuralNetwork(784, [100, 10], [ReLU, softmax], [ReLU_der, identity_der],
                    cross_entropy, softmax_cross_entropy_der, Adam(gamma=0.001))   # a new Adam for the new network

history_net_layer = net_layer.train(x_tr, Y_tr, epochs=5, batch_size=100, rng=np.random.default_rng(2026))

pred_val_net_layer = np.argmax(net_layer.predict(x_val), axis=1)
print("validation accuracy:", accuracy(y_val, pred_val_net_layer))